# 🧭 Supervisor with Agents as Tools

## Learning Objectives
In this notebook, you will learn:
1. **The supervisor pattern** - One central agent decides which specialist works next and combines their results
2. **Specialists** - Build two prebuilt `create_agent` specialists, each with its own tool and data
3. **Agents as tools** - Wrap each specialist in a `@tool`, so the supervisor calls it like any other tool and gets the answer back
4. **Dynamic routing** - Watch the supervisor call one, two or no specialists depending on the request
5. **Step cap** - Stop a run that needs too many steps with `recursion_limit`

## Where this fits
This is pattern 1 of [Five Practical Agent Patterns](../../../../docs/source/08/five-practical-agent-patterns.md). In `01_sequential_and_conditional_workflows.ipynb` the graph fixed every possible path. Here the **supervisor agent** decides at run time which specialists to call, how many times, and when it is done. Every agent is built with LangChain's prebuilt `create_agent`.

**Flow:** `User → supervisor → specialist → supervisor → answer`

---
## 📦 Step 1: Environment Setup

The setup cell imports everything the notebook uses, loads the API keys from the project-root `.env` and sets `RECURSION_LIMIT`, the step cap every run passes (Part 5).

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and imports
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv
from langchain.agents import create_agent
from langchain.tools import tool
from langchain_core.messages import AIMessage, HumanMessage, ToolMessage
from langgraph.errors import GraphRecursionError

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

# Every run passes this cap on LangGraph super-steps (Part 5)
RECURSION_LIMIT = 25

print(f"✅ Environment ready (recursion_limit = {RECURSION_LIMIT})")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: one chat model shared by every agent
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini", temperature=0)

print(f"✅ LLM initialized: {llm.model_name}")

In [ ]:
# ============================================================================
# ASK: run a prebuilt agent on one message and return its final reply
# ============================================================================
def ask(agent, text: str) -> str:
    """Send one user message to a create_agent agent; return the text of its last message."""
    result = agent.invoke({"messages": [HumanMessage(content=text)]}, {"recursion_limit": RECURSION_LIMIT})
    return result["messages"][-1].content


print("✅ ask() ready")

---
## 🧭 Part 1: The Supervisor Pattern

A supervisor is an agent whose tools are **other agents**. It reads the request, decides which specialist should work on it, sends that specialist a task, reads the answer, and repeats until it can reply.

| | Conditional workflow (notebook 01) | Supervisor (this notebook) |
|---|---|---|
| Who picks the next step | a branch function you wrote | the supervisor model, through tool calls |
| Paths | listed in advance | any order and number of specialist calls |
| Calling two specialists for one request | needs a new edge | happens if the request needs it |
| After a specialist answers | the graph moves on | control returns to the supervisor |

There are two common ways to build one. In **graph-based routing** (module 06), the supervisor returns a typed route and graph edges send control to a specialist node. In **agents as tools**, used here, each specialist is wrapped in a tool, so a specialist call is just a tool call and its answer comes back as a `ToolMessage`. No custom graph is needed: the supervisor itself is one `create_agent` agent.

---
## 🧑‍💼 Part 2: Two Specialists With Their Own Tools

A small trip planner with two specialists:

| Specialist | Tool | Data |
|---|---|---|
| `flight_agent` | `search_flights(origin, destination)` | `FLIGHTS`, an in-memory dict |
| `hotel_agent` | `search_hotels(city, max_price_per_night)` | `HOTELS`, an in-memory dict |

The data is fixed, so you can check every answer by eye. Each specialist sees only its own tool: the hotel agent cannot search flights.

In [ ]:
# ============================================================================
# SPECIALIST TOOLS: in-memory flight and hotel data (fares in INR)
# ============================================================================
FLIGHTS = {  # (origin, destination) -> [(flight, departs, fare)]
    ("bangalore", "singapore"): [("SQ 503", "08:10", 21500), ("6E 1007", "23:45", 14800)],
    ("bangalore", "tokyo"): [("SQ 503 + SQ 638", "08:10", 48900), ("AI 306", "21:30", 42300)],
}
HOTELS = {  # city -> [(hotel, price per night)]
    "singapore": [("Marina Budget Inn", 6200), ("Bugis Boutique", 9800), ("Orchard Grand", 18400)],
    "tokyo": [("Shinjuku Capsule Stay", 4100), ("Asakusa Ryokan", 8900), ("Ginza Tower Hotel", 21900)],
}


@tool
def search_flights(origin: str, destination: str) -> str:
    """Search one-way flights between two cities. Returns flight, departure time and fare in INR."""
    options = FLIGHTS.get((origin.strip().lower(), destination.strip().lower()))
    if not options:
        return f"No flights found from {origin} to {destination}."
    return "\n".join(f"{flight} departs {departs}, fare ₹{fare:,}" for flight, departs, fare in options)


@tool
def search_hotels(city: str, max_price_per_night: int = 1_000_000) -> str:
    """Search hotels in a city at or below a nightly price in INR."""
    options = [(name, price) for name, price in HOTELS.get(city.strip().lower(), []) if price <= max_price_per_night]
    if not options:
        return f"No hotels in {city} at or below ₹{max_price_per_night:,} per night."
    return "\n".join(f"{name}: ₹{price:,} per night" for name, price in options)


print(search_flights.invoke({"origin": "Bangalore", "destination": "Singapore"}))
print(search_hotels.invoke({"city": "Tokyo", "max_price_per_night": 9000}))

In [ ]:
# ============================================================================
# SPECIALIST AGENTS: prebuilt create_agent agents, one tool each
# ============================================================================
flight_agent = create_agent(
    llm,
    tools=[search_flights],
    system_prompt=(
        "You are a flight specialist. Always use search_flights before answering. "
        "Recommend one option with its fare in INR. Reply in at most 3 sentences."
    ),
    name="flight_agent",
)
hotel_agent = create_agent(
    llm,
    tools=[search_hotels],
    system_prompt=(
        "You are a hotel specialist. Always use search_hotels before answering. "
        "Recommend one option with its nightly price in INR. Reply in at most 3 sentences."
    ),
    name="hotel_agent",
)

print(ask(hotel_agent, "A hotel in Singapore under ₹10,000 a night, please."))

---
## 🧰 Part 3: Agents as Tools

Each specialist is wrapped in a `@tool`. Inside, the tool runs the specialist with `ask()` and returns its final reply as a string. Two details matter:

- **The docstring is the routing rule.** The supervisor model reads each tool's description to decide which specialist fits. "Use for flights" and "use for hotels" are the only routing logic in this system.
- **The specialist sees only `request`.** It does not see the user's message or the supervisor's history. This **context isolation** keeps each specialist's prompt short, but the supervisor must write a complete request: cities, dates, budget.

In [ ]:
# ============================================================================
# AGENTS AS TOOLS: each specialist wrapped in a @tool
# ============================================================================
@tool
def ask_flight_agent(request: str) -> str:
    """Ask the flight specialist about flights: routes, departure times and fares.
    Pass a complete, self-contained request including both cities and any preferences."""
    return ask(flight_agent, request)


@tool
def ask_hotel_agent(request: str) -> str:
    """Ask the hotel specialist about hotels: options and nightly prices in a city.
    Pass a complete, self-contained request including the city and the nightly budget."""
    return ask(hotel_agent, request)


supervisor = create_agent(
    llm,
    tools=[ask_flight_agent, ask_hotel_agent],
    system_prompt="""You are a travel-planning supervisor. You never search yourself.
- Use ask_flight_agent for flight questions and ask_hotel_agent for hotel questions.
- Call only the specialists the request needs. Give each one a complete request.
- If the request needs no specialist, answer it directly.
- Combine the specialists' answers into one short plan. When both a flight and a hotel are chosen,
  give the total cost in INR (flight fare + nights x nightly price).""",
    name="supervisor",
)

print("✅ Supervisor ready with tools:", [t.name for t in [ask_flight_agent, ask_hotel_agent]])

---
## ▶️ Part 4: Dynamic Routing

`run_supervisor` sends one request and reads the trace from the messages: each `AIMessage` with `tool_calls` is a supervisor decision, and each `ToolMessage` is a specialist's answer coming back. Control returns to the supervisor after every specialist call.

Three requests need different specialists:
1. A full trip → **both** specialists
2. Hotels only → **only** `ask_hotel_agent`
3. A question about the planner itself → **no** specialist

In [ ]:
# ============================================================================
# RUN_SUPERVISOR: send one request and show every delegation
# ============================================================================
def run_supervisor(request: str) -> list[str]:
    """Run the supervisor once; print each delegation and the final answer; return the tools it called."""
    result = supervisor.invoke({"messages": [HumanMessage(content=request)]}, {"recursion_limit": RECURSION_LIMIT})
    called = []
    print("=" * 80)
    print(f"USER: {request}")
    for message in result["messages"]:
        if isinstance(message, AIMessage):
            for call in message.tool_calls:
                called.append(call["name"])
                print(f"\n🧭 supervisor → {call['name']}: {call['args']['request']}")
        elif isinstance(message, ToolMessage):
            print(f"↩️ {message.name} answered: {message.content}")
    print(f"\n💬 FINAL: {result['messages'][-1].content}")
    return called


REQUESTS = [  # (request, specialists the supervisor should call)
    ("Plan a 3-night trip from Bangalore to Singapore: the cheapest flight and a hotel under ₹10,000 a night.",
     {"ask_flight_agent", "ask_hotel_agent"}),
    ("Which hotels in Tokyo cost less than ₹9,000 a night?", {"ask_hotel_agent"}),
    ("What kinds of questions can you help me with?", set()),
]

passed = 0
for request, expected in REQUESTS:
    called = run_supervisor(request)
    ok = set(called) == expected
    passed += ok
    print(f"{'✅' if ok else '❌'} called {sorted(set(called)) or 'no specialist'}; expected {sorted(expected) or 'none'}")

print(f"\n{passed}/{len(REQUESTS)} requests reached the expected specialists")

Read the 🧭 lines: they are the requests the supervisor *wrote* for each specialist. In the full trip, a good supervisor passes the budget to the hotel agent, because the hotel agent never sees the user's message. The ↩️ lines show each answer returning to the supervisor, which then writes the combined plan. No edge or branch function decided any of this; the supervisor's tool calls did.

---
## 🛡️ Part 5: Cap the Number of Steps

A supervisor can keep calling specialists if it is never satisfied. Each model call and each tool round is one LangGraph super-step, so `recursion_limit` in the run config caps the whole run. When the cap is reached, LangGraph raises `GraphRecursionError` instead of running on.

The full-trip request needs at least three super-steps: the supervisor's model call, the specialist tool calls, and a second model call to write the answer. With `recursion_limit=2` the run must stop with the error, caught in code below.

In [ ]:
# ============================================================================
# STEP CAP: a run that needs more super-steps than allowed
# ============================================================================
try:
    supervisor.invoke({"messages": [HumanMessage(content=REQUESTS[0][0])]}, {"recursion_limit": 2})
    print("❌ the run finished inside 2 super-steps")
except GraphRecursionError as error:
    print(f"✅ GraphRecursionError: {str(error).splitlines()[0]}")

Every other run in this notebook passes `RECURSION_LIMIT = 25`: enough room for several delegations, small enough that a confused supervisor fails fast.

---
## 📝 Summary

In this notebook, we learned:

### 1. The supervisor pattern
- A central agent picks the specialists at run time, and control returns to it after each one. Unlike a conditional workflow, the set of steps is not listed in advance.

### 2. Specialists
- Each specialist is a prebuilt `create_agent` agent with its own tool and prompt; it cannot use the other specialist's tool.

### 3. Agents as tools
- A `@tool` that runs a specialist turns a delegation into an ordinary tool call; the answer comes back as a `ToolMessage`.
- The tool docstring is the routing rule, and the specialist sees only the request the supervisor writes (context isolation).

### 4. Dynamic routing
- One supervisor called both specialists, only one, or none, depending on the request, without any branch function.

### 5. Step cap
- `recursion_limit` in the run config turns a run that needs too many super-steps into a `GraphRecursionError`.

### Next Steps
- **`03_parallel_fan_out_fan_in.ipynb`**: run independent agents at the same time and merge their results